In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "src" / "db.py").exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("src/db.py tapılmadı")
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
from src.db import get_engine
from sqlalchemy import text

engine = get_engine()

---------------------------------------------------------------------------------------

# Müştəri və churn analitikası


### Ümumi churn faizi

In [4]:
from sqlalchemy import text
import pandas as pd

churn_df = pd.read_sql(text("""
    SELECT
        churned,
        COUNT(DISTINCT customer_id) AS musteri_sayi,
        ROUND(
            COUNT(DISTINCT customer_id) * 100.0 /
            (SELECT COUNT(DISTINCT customer_id) FROM silver.loans),
            2
        ) AS churn_faizi
    FROM silver.loans
    GROUP BY churned
    ORDER BY churned
"""), engine)

display(churn_df)

churn_df.to_sql(
    "churn_report",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

,churned,musteri_sayi,churn_faizi
0,0,3992,79.84
1,1,1008,20.16


2

### Aktivlik üzrə churn

In [5]:
activity_df = pd.read_sql("""
    SELECT
        CASE WHEN is_active = 1 THEN N'Aktivdir'
             ELSE N'Aktiv deyil' END AS Aktivlik,
        COUNT(*) AS musteri_sayi,
        SUM(churned) AS geden_musteri,
        ROUND(AVG(CAST(churned AS FLOAT)) * 100, 2) AS churn_faizi
    FROM (
        SELECT DISTINCT customer_id, is_active, churned
        FROM silver.loans
    ) AS customers
    GROUP BY is_active
    ORDER BY is_active
""", engine)

display(activity_df)

,Aktivlik,musteri_sayi,geden_musteri,churn_faizi
0,Aktiv deyil,2029,584,28.78
1,Aktivdir,2971,424,14.27


In [6]:
activity_df.to_sql(
    "churn_by_activity",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

2

### 3 MƏHSUL SAYINA GÖRƏ CHURN ANALİZİ

In [7]:
product_df = pd.read_sql("""
    SELECT
        products_count,
        COUNT(*) AS musteri_sayi,
        SUM(churned) AS geden_musteri,
        ROUND(AVG(CAST(churned AS FLOAT)) * 100, 2) AS churn_faizi
    FROM (
        SELECT DISTINCT customer_id, products_count, churned
        FROM silver.loans
    ) AS customers
    GROUP BY products_count
    ORDER BY products_count
""", engine)

display(product_df)

,products_count,musteri_sayi,geden_musteri,churn_faizi
0,1,2289,520,22.72
1,2,2114,214,10.12
2,3,441,171,38.78
3,4,156,103,66.03


In [8]:
product_df.to_sql(
    "churn_by_products",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

4

### 4. MƏHSUL SAYI VƏ AKTİVLİYƏ GÖRƏ CHURN ANALİZİ

In [9]:
product_activity_df = pd.read_sql("""
    SELECT
        products_count,
        CASE WHEN is_active = 1 THEN N'Aktivdir'
             ELSE N'Aktiv deyil' END AS Aktivlik,
        COUNT(*) AS musteri_sayi,
        SUM(churned) AS geden_musteri,
        ROUND(AVG(CAST(churned AS FLOAT)) * 100, 2) AS churn_faizi
    FROM (
        SELECT DISTINCT customer_id, products_count, is_active, churned
        FROM silver.loans
    ) AS customers
    GROUP BY products_count, is_active
    ORDER BY products_count, is_active
""", engine)

display(product_activity_df)

,products_count,Aktivlik,musteri_sayi,geden_musteri,churn_faizi
0,1,Aktiv deyil,1037,328,31.63
1,1,Aktivdir,1252,192,15.34
2,2,Aktiv deyil,787,133,16.90
3,2,Aktivdir,1327,81,6.10
4,3,Aktiv deyil,146,75,51.37
5,3,Aktivdir,295,96,32.54
6,4,Aktiv deyil,59,48,81.36
7,4,Aktivdir,97,55,56.70


In [10]:
product_activity_df.to_sql(
    "churn_by_product_activity",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

8

### 5. ƏMƏLİYYAT SAYINA GÖRƏ CHURN ANALİZİ

In [11]:
transaction_churn_df = pd.read_sql("""
    SELECT
        CASE
            WHEN transaction_count <= 5 THEN '1-5'
            WHEN transaction_count <= 10 THEN '6-10'
            WHEN transaction_count <= 20 THEN '11-20'
            ELSE '21+'
        END AS transaction_group,
        COUNT(*) AS musteri_sayi,
        SUM(churned) AS geden_musteri,
        ROUND(AVG(CAST(churned AS FLOAT)) * 100, 2) AS churn_faizi
    FROM (
        SELECT
            customer_id,
            MAX(churned) AS churned,
            COUNT(transaction_id) AS transaction_count
        FROM silver.transactions
        GROUP BY customer_id
    ) AS customers
    GROUP BY
        CASE
            WHEN transaction_count <= 5 THEN '1-5'
            WHEN transaction_count <= 10 THEN '6-10'
            WHEN transaction_count <= 20 THEN '11-20'
            ELSE '21+'
        END
    ORDER BY MIN(transaction_count)
""", engine)

display(transaction_churn_df)

,transaction_group,musteri_sayi,geden_musteri,churn_faizi
0,1-5,884,332,37.56
1,6-10,1062,246,23.16
2,11-20,1266,274,21.64
3,21+,1785,154,8.63


In [12]:
transaction_churn_df.to_sql(
    "churn_by_transaction_count",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

4

### 6. ƏMƏLİYYAT KANALLARINA GÖRƏ CHURN ANALİZİ

In [13]:
channel_df = pd.read_sql("""
    SELECT
        channel AS Kanal,
        COUNT(DISTINCT customer_id) AS musteri_sayi,
        COUNT(DISTINCT CASE
            WHEN churned = 1 THEN customer_id
        END) AS geden_musteri,
        ROUND(
            COUNT(DISTINCT CASE
                WHEN churned = 1 THEN customer_id
            END) * 100.0 /
            COUNT(DISTINCT customer_id), 2
        ) AS churn_faizi
    FROM silver.transactions
    GROUP BY channel
    ORDER BY churn_faizi DESC
""", engine)

display(channel_df)

,Kanal,musteri_sayi,geden_musteri,churn_faizi
0,mobil tətbiq,4847,933,19.25
1,pos,4617,878,19.02
2,atm,4313,789,18.29
3,filial,3396,582,17.14


In [14]:
channel_df.to_sql(
    "churn_by_channel",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

4

### 7. YAŞ QRUPLARINA GÖRƏ CHURN ANALİZİ

In [15]:
age_df = pd.read_sql("""
    SELECT
        CASE
            WHEN age <= 25 THEN '18-25'
            WHEN age <= 35 THEN '26-35'
            WHEN age <= 45 THEN '36-45'
            WHEN age <= 55 THEN '46-55'
            ELSE '56+'
        END AS yas_qrupu,
        COUNT(*) AS musteri_sayi,
        SUM(churned) AS geden_musteri,
        ROUND(AVG(CAST(churned AS FLOAT)) * 100, 2) AS churn_faizi
    FROM (
        SELECT DISTINCT customer_id, age, churned
        FROM silver.loans
    ) AS customers
    GROUP BY
        CASE
            WHEN age <= 25 THEN '18-25'
            WHEN age <= 35 THEN '26-35'
            WHEN age <= 45 THEN '36-45'
            WHEN age <= 55 THEN '46-55'
            ELSE '56+'
        END
    ORDER BY MIN(age)
""", engine)

display(age_df)

,yas_qrupu,musteri_sayi,geden_musteri,churn_faizi
0,18-25,702,94,13.39
1,26-35,1279,201,15.72
2,36-45,1572,297,18.89
3,46-55,1015,272,26.80
4,56+,432,144,33.33


In [16]:
age_df.to_sql(
    "churn_by_age",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

5

### 8. BANKDA KEÇİRİLƏN MÜDDƏTƏ GÖRƏ CHURN ANALİZİ

In [17]:
tenure_df = pd.read_sql("""
    SELECT
        CASE
            WHEN tenure_years <= 2 THEN '0-2 il'
            WHEN tenure_years <= 5 THEN '3-5 il'
            WHEN tenure_years <= 8 THEN '6-8 il'
            ELSE '9+ il'
        END AS muddet_qrupu,
        COUNT(*) AS musteri_sayi,
        SUM(churned) AS geden_musteri,
        ROUND(AVG(CAST(churned AS FLOAT)) * 100, 2) AS churn_faizi
    FROM (
        SELECT DISTINCT customer_id, tenure_years, churned
        FROM silver.loans
    ) AS customers
    GROUP BY
        CASE
            WHEN tenure_years <= 2 THEN '0-2 il'
            WHEN tenure_years <= 5 THEN '3-5 il'
            WHEN tenure_years <= 8 THEN '6-8 il'
            ELSE '9+ il'
        END
    ORDER BY MIN(tenure_years)
""", engine)

display(tenure_df)

,muddet_qrupu,musteri_sayi,geden_musteri,churn_faizi
0,0-2 il,1441,305,21.17
1,3-5 il,1868,390,20.88
2,6-8 il,1046,204,19.50
3,9+ il,645,109,16.90


In [18]:
tenure_df.to_sql(
    "churn_by_tenure",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

4

### 9. MÜŞTƏRİ BALANSINA GÖRƏ CHURN ANALİZİ

In [19]:
balance_df = pd.read_sql("""
    SELECT
        CASE
            WHEN balance = 0 THEN '0 AZN'
            WHEN balance <= 10000 THEN '1-10000 AZN'
            WHEN balance <= 50000 THEN '10001-50000 AZN'
            WHEN balance <= 100000 THEN '50001-100000 AZN'
            ELSE '100000+ AZN'
        END AS balans_qrupu,
        COUNT(*) AS musteri_sayi,
        SUM(churned) AS geden_musteri,
        ROUND(AVG(CAST(churned AS FLOAT)) * 100, 2) AS churn_faizi
    FROM (
        SELECT DISTINCT customer_id, balance, churned
        FROM silver.loans
        WHERE balance IS NOT NULL
    ) AS customers
    GROUP BY
        CASE
            WHEN balance = 0 THEN '0 AZN'
            WHEN balance <= 10000 THEN '1-10000 AZN'
            WHEN balance <= 50000 THEN '10001-50000 AZN'
            WHEN balance <= 100000 THEN '50001-100000 AZN'
            ELSE '100000+ AZN'
        END
    ORDER BY MIN(balance)
""", engine)

display(balance_df)

,balans_qrupu,musteri_sayi,geden_musteri,churn_faizi
0,0 AZN,616,121,19.64
1,1-10000 AZN,3424,664,19.39
2,10001-50000 AZN,912,212,23.25
3,50001-100000 AZN,43,10,23.26
4,100000+ AZN,5,1,20.00


In [20]:
balance_df.to_sql(
    "churn_by_balance",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

5

### 10. MAAŞ SƏVİYYƏSİNƏ GÖRƏ CHURN ANALİZİ

In [21]:
salary_df = pd.read_sql("""
    SELECT
        CASE
            WHEN salary <= 1000 THEN '0-1000 AZN'
            WHEN salary <= 2000 THEN '1001-2000 AZN'
            WHEN salary <= 3000 THEN '2001-3000 AZN'
            WHEN salary <= 5000 THEN '3001-5000 AZN'
            ELSE '5000+ AZN'
        END AS maas_qrupu,
        COUNT(*) AS musteri_sayi,
        SUM(churned) AS geden_musteri,
        ROUND(AVG(CAST(churned AS FLOAT)) * 100, 2) AS churn_faizi
    FROM (
        SELECT DISTINCT customer_id, salary, churned
        FROM silver.loans
        WHERE salary IS NOT NULL
    ) AS customers
    GROUP BY
        CASE
            WHEN salary <= 1000 THEN '0-1000 AZN'
            WHEN salary <= 2000 THEN '1001-2000 AZN'
            WHEN salary <= 3000 THEN '2001-3000 AZN'
            WHEN salary <= 5000 THEN '3001-5000 AZN'
            ELSE '5000+ AZN'
        END
    ORDER BY MIN(salary)
""", engine)

display(salary_df)

,maas_qrupu,musteri_sayi,geden_musteri,churn_faizi
0,0-1000 AZN,1809,359,19.85
1,1001-2000 AZN,2255,441,19.56
2,2001-3000 AZN,654,139,21.25
3,3001-5000 AZN,254,61,24.02
4,5000+ AZN,28,8,28.57


In [22]:
salary_df.to_sql(
    "churn_by_salary",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

5

### 11. KREDİT SKORUNA GÖRƏ CHURN ANALİZİ

In [23]:
credit_df = pd.read_sql("""
    SELECT
        CASE
            WHEN credit_score < 500 THEN '300-499'
            WHEN credit_score < 600 THEN '500-599'
            WHEN credit_score < 700 THEN '600-699'
            WHEN credit_score < 800 THEN '700-799'
            ELSE '800+'
        END AS kredit_skoru_qrupu,
        COUNT(*) AS musteri_sayi,
        SUM(churned) AS geden_musteri,
        ROUND(AVG(CAST(churned AS FLOAT)) * 100, 2) AS churn_faizi
    FROM (
        SELECT DISTINCT customer_id, credit_score, churned
        FROM silver.loans
        WHERE credit_score IS NOT NULL
    ) AS customers
    GROUP BY
        CASE
            WHEN credit_score < 500 THEN '300-499'
            WHEN credit_score < 600 THEN '500-599'
            WHEN credit_score < 700 THEN '600-699'
            WHEN credit_score < 800 THEN '700-799'
            ELSE '800+'
        END
    ORDER BY MIN(credit_score)
""", engine)

display(credit_df)

,kredit_skoru_qrupu,musteri_sayi,geden_musteri,churn_faizi
0,300-499,143,56,39.16
1,500-599,1063,302,28.41
2,600-699,2168,433,19.97
3,700-799,1361,179,13.15
4,800+,265,38,14.34


In [24]:
credit_df.to_sql(
    "churn_by_credit_score",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

5

### 12 KREDİT MÜRACİƏTİNİN RƏDD EDİLMƏSİNƏ GÖRƏ CHURN ANALİZİ

In [25]:
rejection_df = pd.read_sql("""
    SELECT
        kredit_statusu,
        COUNT(*) AS musteri_sayi,
        SUM(churned) AS geden_musteri,
        ROUND(AVG(CAST(churned AS FLOAT)) * 100, 2) AS churn_faizi
    FROM (
        SELECT
            customer_id,
            MAX(churned) AS churned,
            CASE
                WHEN MAX(CASE WHEN status = N'Rədd edildi'
                              THEN 1 ELSE 0 END) = 1
                THEN N'Rədd edilib'
                ELSE N'Təsdiqlənib'
            END AS kredit_statusu
        FROM silver.loans
        WHERE status IS NOT NULL
        GROUP BY customer_id
    ) AS customers
    GROUP BY kredit_statusu
    ORDER BY kredit_statusu
""", engine)

display(rejection_df)

,kredit_statusu,musteri_sayi,geden_musteri,churn_faizi
0,Rədd edilib,1461,192,13.14
1,Təsdiqlənib,1463,186,12.71


In [26]:
rejection_df.to_sql(
    "churn_by_loan_rejection",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

2

### 13 Şəhərlər üzrə churn analizi

In [27]:
city_df = pd.read_sql("""
    SELECT
        city AS seher,
        COUNT(*) AS musteri_sayi,
        SUM(churned) AS geden_musteri,
        ROUND(AVG(CAST(churned AS FLOAT)) * 100, 2) AS churn_faizi
    FROM (
        SELECT DISTINCT customer_id, city, churned
        FROM silver.loans
    ) AS customers
    GROUP BY city
    ORDER BY churn_faizi DESC
""", engine)

display(city_df)

,seher,musteri_sayi,geden_musteri,churn_faizi
0,mingəçevir,285,67,23.51
1,lənkəran,289,64,22.15
2,gəncə,640,138,21.56
3,şəki,266,56,21.05
4,bakı,2805,554,19.75
5,sumqayıt,474,86,18.14
6,naxçıvan,241,43,17.84


In [28]:
city_df.to_sql(
    "churn_by_city",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

7

### 14 Gender üzrə churn analizi

In [29]:
gender_df = pd.read_sql("""
    SELECT
        gender AS cinsiyyet,
        COUNT(*) AS musteri_sayi,
        SUM(churned) AS geden_musteri,
        ROUND(AVG(CAST(churned AS FLOAT)) * 100, 2) AS churn_faizi
    FROM (
        SELECT DISTINCT customer_id, gender, churned
        FROM silver.loans
    ) AS customers
    GROUP BY gender
    ORDER BY churn_faizi DESC
""", engine)

display(gender_df)

,cinsiyyet,musteri_sayi,geden_musteri,churn_faizi
0,qadın,2413,539,22.34
1,kişi,2587,469,18.13


In [30]:
gender_df.to_sql(
    "churn_by_gender",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

2

### 15 Filiallar üzrə Churn Analizi (Branch)

In [31]:
branch_df = pd.read_sql("""
    SELECT
        branch AS filial,
        COUNT(*) AS musteri_sayi,
        SUM(churned) AS geden_musteri,
        ROUND(AVG(CAST(churned AS FLOAT)) * 100, 2) AS churn_faizi
    FROM (
        SELECT DISTINCT customer_id, branch, churned
        FROM silver.loans
    ) AS customers
    GROUP BY branch
    ORDER BY churn_faizi DESC
""", engine)

display(branch_df)

,filial,musteri_sayi,geden_musteri,churn_faizi
0,mingəçevir mərkəz,285,67,23.51
1,nərimanov,459,105,22.88
2,lənkəran mərkəz,289,64,22.15
3,gəncə mərkəz,640,138,21.56
4,yasamal,457,98,21.44
5,şəki mərkəz,266,56,21.05
6,nəsimi,496,101,20.36
7,xətai,477,93,19.50
8,binəqədi,460,86,18.70
9,sumqayıt mərkəz,474,86,18.14


In [32]:
branch_df.to_sql(
    "churn_by_branch",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

12

# Kredit müraciətlərində satış itkisi

### Neçə müraciət təsdiqlənib, neçəsi rədd edilib?

In [33]:
loan_status_df = pd.read_sql("""
    SELECT
        status AS kredit_statusu,
        COUNT(loan_id) AS muraciet_sayi,
        ROUND(
            COUNT(loan_id) * 100.0 /
            (SELECT COUNT(loan_id) FROM silver.loans),
            2
        ) AS faiz
    FROM silver.loans
    WHERE loan_id IS NOT NULL
    GROUP BY status
    ORDER BY muraciet_sayi DESC
""", engine)

display(loan_status_df)

,kredit_statusu,muraciet_sayi,faiz
0,təsdiqləndi,2975,61.98
1,rədd edildi,1825,38.02


In [34]:
loan_status_df.to_sql(
    "loan_application_status",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

2

### Rədd edilmiş və təsdiqlənmiş müraciətlərin ümumi məbləği nə qədərdir?

In [54]:
loan_amount_status_df = pd.read_sql("""
    SELECT
        status AS kredit_statusu,

        COUNT(loan_id) AS muraciet_sayi,

        SUM(amount) AS umumi_mebleg,

        ROUND(AVG(amount), 2) AS orta_mebleg

    FROM silver.loans
    WHERE loan_id IS NOT NULL
      AND amount IS NOT NULL
      AND status IN (N'Rədd edildi', N'Təsdiqləndi')

    GROUP BY status

    ORDER BY
        CASE
            WHEN status = N'Rədd edildi' THEN 1
            WHEN status = N'Təsdiqləndi' THEN 2
        END
""", engine)

display(loan_amount_status_df)

,kredit_statusu,muraciet_sayi,umumi_mebleg,orta_mebleg
0,rədd edildi,1825,1.707945e+08,93586.01
1,təsdiqləndi,2975,1.298844e+08,43658.61


In [55]:
loan_amount_status_df.to_sql(
    "loan_amount_by_status",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

2

### Ən çox hansı səbəbdən müraciətlər rədd edilir?

In [35]:
rejection_reason_df = pd.read_sql("""
    SELECT
        rejection_reason AS redd_sebebi,
        COUNT(loan_id) AS muraciet_sayi,
        ROUND(
            COUNT(loan_id) * 100.0 /
            (SELECT COUNT(loan_id)
             FROM silver.loans
             WHERE status = N'Rədd edildi'),
            2
        ) AS faiz
    FROM silver.loans
    WHERE status = N'Rədd edildi'
    GROUP BY rejection_reason
    ORDER BY muraciet_sayi DESC
""", engine)

display(rejection_reason_df)

,redd_sebebi,muraciet_sayi,faiz
0,yüksək borc yükü,1227,67.23
1,aşağı kredit skoru,311,17.04
2,Səbəb qeyd olunmayıb,91,4.99
3,qeyri-kafi gəlir sənədi,75,4.11
4,natamam sənədlər,62,3.40
5,bank siyasəti,59,3.23


In [36]:
rejection_reason_df.to_sql(
    "loan_rejection_reasons",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

6

### Hansı kredit növü daha çox rədd edilir?

In [50]:
product_rejection_df = pd.read_sql("""
    SELECT
        product_type AS kredit_novu,

        COUNT(loan_id) AS muraciet_sayi,

        SUM(CASE WHEN status = N'Rədd edildi'
            THEN 1 ELSE 0 END) AS redd_edilen,

        ROUND(
            SUM(CASE WHEN status = N'Rədd edildi'
                THEN 1 ELSE 0 END) * 100.0 / COUNT(loan_id),
            2
        ) AS redd_faizi

    FROM silver.loans
    WHERE loan_id IS NOT NULL
      AND product_type IS NOT NULL

    GROUP BY product_type
    ORDER BY redd_faizi DESC
""", engine)

display(product_rejection_df)

,kredit_novu,muraciet_sayi,redd_edilen,redd_faizi
0,biznes krediti,866,637,73.56
1,ipoteka,553,333,60.22
2,avtomobil krediti,711,318,44.73
3,istehlak krediti,2670,537,20.11


In [51]:
product_rejection_df.to_sql(
    "loan_rejection_by_product",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

4

### Kredit skoru aşağı olan müştərilərin müraciətləri daha çox rədd edilirmi?

In [37]:
credit_rejection_df = pd.read_sql("""
    SELECT
        CASE
            WHEN credit_score < 500 THEN '300-499'
            WHEN credit_score < 600 THEN '500-599'
            WHEN credit_score < 700 THEN '600-699'
            WHEN credit_score < 800 THEN '700-799'
            ELSE '800+'
        END AS kredit_skoru_qrupu,

        COUNT(loan_id) AS muraciet_sayi,

        SUM(CASE WHEN status = N'Rədd edildi'
            THEN 1 ELSE 0 END) AS redd_edilen,

        ROUND(
            SUM(CASE WHEN status = N'Rədd edildi'
                THEN 1 ELSE 0 END) * 100.0 / COUNT(loan_id),
            2
        ) AS redd_faizi

    FROM silver.loans
    WHERE loan_id IS NOT NULL
      AND credit_score IS NOT NULL
    GROUP BY
        CASE
            WHEN credit_score < 500 THEN '300-499'
            WHEN credit_score < 600 THEN '500-599'
            WHEN credit_score < 700 THEN '600-699'
            WHEN credit_score < 800 THEN '700-799'
            ELSE '800+'
        END
    ORDER BY MIN(credit_score)
""", engine)

display(credit_rejection_df)

,kredit_skoru_qrupu,muraciet_sayi,redd_edilen,redd_faizi
0,300-499,97,77,79.38
1,500-599,878,527,60.02
2,600-699,2049,774,37.77
3,700-799,1477,394,26.68
4,800+,299,53,17.73


In [38]:
credit_rejection_df.to_sql(
    "loan_rejection_by_credit_score",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

5

### Hansı yaş qrupunun müraciətləri daha çox təsdiqlənir?

In [39]:
age_approval_df = pd.read_sql("""
    SELECT
        CASE
            WHEN age <= 25 THEN '18-25'
            WHEN age <= 35 THEN '26-35'
            WHEN age <= 45 THEN '36-45'
            WHEN age <= 55 THEN '46-55'
            ELSE '56+'
        END AS yas_qrupu,

        COUNT(loan_id) AS muraciet_sayi,

        SUM(CASE WHEN status = N'Təsdiqləndi'
            THEN 1 ELSE 0 END) AS tesdiqlenen,

        ROUND(
            SUM(CASE WHEN status = N'Təsdiqləndi'
                THEN 1 ELSE 0 END) * 100.0 / COUNT(loan_id),
            2
        ) AS tesdiq_faizi

    FROM silver.loans
    WHERE loan_id IS NOT NULL
      AND age IS NOT NULL

    GROUP BY
        CASE
            WHEN age <= 25 THEN '18-25'
            WHEN age <= 35 THEN '26-35'
            WHEN age <= 45 THEN '36-45'
            WHEN age <= 55 THEN '46-55'
            ELSE '56+'
        END

    ORDER BY MIN(age)
""", engine)

display(age_approval_df)

,yas_qrupu,muraciet_sayi,tesdiqlenen,tesdiq_faizi
0,18-25,671,418,62.30
1,26-35,1261,786,62.33
2,36-45,1500,909,60.60
3,46-55,974,594,60.99
4,56+,394,268,68.02


In [40]:
age_approval_df.to_sql(
    "loan_approval_by_age",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

5

### Hansı filiallarda rədd faizi daha yüksəkdir?

In [41]:
branch_rejection_df = pd.read_sql("""
    SELECT
        branch AS filial,
        COUNT(loan_id) AS muraciet_sayi,
        SUM(CASE WHEN status = N'Rədd edildi'
            THEN 1 ELSE 0 END) AS redd_edilen,
        ROUND(
            SUM(CASE WHEN status = N'Rədd edildi'
                THEN 1 ELSE 0 END) * 100.0 / COUNT(loan_id),
            2
        ) AS redd_faizi
    FROM silver.loans
    WHERE loan_id IS NOT NULL
      AND branch IS NOT NULL
    GROUP BY branch
    ORDER BY redd_faizi DESC
""", engine)

display(branch_rejection_df)

,filial,muraciet_sayi,redd_edilen,redd_faizi
0,şəki mərkəz,262,113,43.13
1,nərimanov,425,178,41.88
2,xətai,444,183,41.22
3,sumqayıt mərkəz,441,178,40.36
4,gəncə mərkəz,615,248,40.33
5,lənkəran mərkəz,284,111,39.08
6,naxçıvan mərkəz,237,89,37.55
7,mingəçevir mərkəz,286,107,37.41
8,səbail,458,165,36.03
9,yasamal,433,154,35.57


In [42]:
branch_rejection_df.to_sql(
    "loan_rejection_by_branch",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

12

### Hansı şəhərlərdə kreditə tələb və rədd faizi yüksəkdir?

In [43]:
city_loan_df = pd.read_sql("""
    SELECT
        city AS seher,
        COUNT(loan_id) AS muraciet_sayi,
        SUM(CASE WHEN status = N'Təsdiqləndi'
            THEN 1 ELSE 0 END) AS tesdiqlenen,
        SUM(CASE WHEN status = N'Rədd edildi'
            THEN 1 ELSE 0 END) AS redd_edilen,
        ROUND(
            SUM(CASE WHEN status = N'Rədd edildi'
                THEN 1 ELSE 0 END) * 100.0 / COUNT(loan_id),
            2
        ) AS redd_faizi
    FROM silver.loans
    WHERE loan_id IS NOT NULL
      AND city IS NOT NULL
    GROUP BY city
    ORDER BY muraciet_sayi DESC
""", engine)

display(city_loan_df)

,seher,muraciet_sayi,tesdiqlenen,redd_edilen,redd_faizi
0,bakı,2675,1696,979,36.60
1,gəncə,615,367,248,40.33
2,sumqayıt,441,263,178,40.36
3,mingəçevir,286,179,107,37.41
4,lənkəran,284,173,111,39.08
5,şəki,262,149,113,43.13
6,naxçıvan,237,148,89,37.55


In [44]:
city_loan_df.to_sql(
    "loan_demand_rejection_by_city",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

7

### Böyük məbləğli müraciətlər daha çox rədd edilirmi?

In [46]:
loan_amount_df = pd.read_sql("""
    SELECT
        CASE
            WHEN amount <= 5000 THEN '0-5000 AZN'
            WHEN amount <= 10000 THEN '5001-10000 AZN'
            WHEN amount <= 20000 THEN '10001-20000 AZN'
            WHEN amount <= 50000 THEN '20001-50000 AZN'
            ELSE '50000+ AZN'
        END AS mebleg_qrupu,

        COUNT(loan_id) AS muraciet_sayi,

        SUM(CASE WHEN status = N'Rədd edildi'
            THEN 1 ELSE 0 END) AS redd_edilen,

        ROUND(
            SUM(CASE WHEN status = N'Rədd edildi'
                THEN 1 ELSE 0 END) * 100.0 / COUNT(loan_id),
            2
        ) AS redd_faizi

    FROM silver.loans
    WHERE loan_id IS NOT NULL
      AND amount IS NOT NULL

    GROUP BY
        CASE
            WHEN amount <= 5000 THEN '0-5000 AZN'
            WHEN amount <= 10000 THEN '5001-10000 AZN'
            WHEN amount <= 20000 THEN '10001-20000 AZN'
            WHEN amount <= 50000 THEN '20001-50000 AZN'
            ELSE '50000+ AZN'
        END

    ORDER BY MIN(amount)
""", engine)

display(loan_amount_df)

,mebleg_qrupu,muraciet_sayi,redd_edilen,redd_faizi
0,0-5000 AZN,1479,226,15.28
1,5001-10000 AZN,1063,257,24.18
2,10001-20000 AZN,686,288,41.98
3,20001-50000 AZN,960,602,62.71
4,50000+ AZN,612,452,73.86


In [47]:
loan_amount_df.to_sql(
    "loan_rejection_by_amount",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

5

### 	Kredit müraciətləri və təsdiq faizi zamanla necə dəyişir?

In [48]:
monthly_loan_df = pd.read_sql("""
    SELECT
        YEAR(application_date) AS il,
        MONTH(application_date) AS ay,

        COUNT(loan_id) AS muraciet_sayi,

        SUM(CASE WHEN status = N'Təsdiqləndi'
            THEN 1 ELSE 0 END) AS tesdiqlenen,

        SUM(CASE WHEN status = N'Rədd edildi'
            THEN 1 ELSE 0 END) AS redd_edilen,

        ROUND(
            SUM(CASE WHEN status = N'Təsdiqləndi'
                THEN 1 ELSE 0 END) * 100.0 / COUNT(loan_id),
            2
        ) AS tesdiq_faizi

    FROM silver.loans
    WHERE loan_id IS NOT NULL
      AND application_date IS NOT NULL

    GROUP BY
        YEAR(application_date),
        MONTH(application_date)

    ORDER BY il, ay
""", engine)

display(monthly_loan_df)

,il,ay,muraciet_sayi,tesdiqlenen,redd_edilen,tesdiq_faizi
0,2026,1,848,519,329,61.20
1,2026,2,965,602,363,62.38
2,2026,3,599,364,235,60.77
3,2026,4,873,527,346,60.37
4,2026,5,748,471,277,62.97
5,2026,6,767,492,275,64.15


In [49]:
monthly_loan_df.to_sql(
    "loan_monthly_trend",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

6

# 

## Bayram günlərində əməliyyat sayı və məbləği dəyişirmi?

##	Müştərilər daha çox hansı günlərdə əməliyyat edir?

## Bayram ərəfəsində xərclər artırmı?

## AZN, USD və EUR əməliyyatlarının sayı və həcmi necə dəyişir?

## Yağışlı, küləkli və isti günlərdə aktivlik fərqlənirmi?

## Bayram günlərində hansı xərc kateqoriyaları artır?